## This is the python notebook of the orthophoto analysis of Khirbet al-Fakheit between 2021 and 2023 for GeoHumanitarian Action class, University of Salzburg

###### This notebook was created with support of AI
---



In [ ]:
!pip -q install rasterio geopandas pyproj requests

In [ ]:
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

from pyproj import Transformer
from pathlib import Path

# GeoMOLG REST API
base_url = "https://orthophotos.geomolg.ps/adaptor/rest/services"

services = {
    2021: f"{base_url}/Orthophotos_WB_2021_15cm_tif_PG1923/MapServer",
    2022: f"{base_url}/Orthophotos_WB_2022_15cm_tif_PG1923_05/MapServer",
    2023: f"{base_url}/Orthophotos_WB_2023_15cm_jp2_PG1923_jp2/MapServer"
}

# Create data directory
data_dir = Path("/content/orthophotos")
data_dir.mkdir(exist_ok=True)

# Check data availability
for year, url in services.items():
    try:
        response = requests.get(
            url,
            params={"f": "pjson"},
            timeout=30
        )
        response.raise_for_status()
        metadata = response.json()

        if "error" in metadata:
            print(f"{year}: {metadata['error']}")
        else:
            print(f"{year}: Available")
            print(f"  CRS: {metadata.get('spatialReference')}")
            print(f"  Max image width: {metadata.get('maxImageWidth')}")
            print(f"  Max image height: {metadata.get('maxImageHeight')}")

    except Exception as e:
        print(f"{year}: Error - {e}")

In [ ]:
# Define AOI

from shapely.geometry import box

# Approximate centre of Khirbet al-Fakheit
lat = 31.37662
lon = 35.16619

# Transform coordinates to the GeoMOLG CRS
transformer = Transformer.from_crs(
    "EPSG:4326", "EPSG:28191", always_xy=True
)

x, y = transformer.transform(lon, lat)

# Define a 1 x 1 km study area
aoi_size = 1000
half_size = aoi_size / 2

xmin = x - half_size
ymin = y - half_size
xmax = x + half_size
ymax = y + half_size

bbox = (xmin, ymin, xmax, ymax)

# Create AOI polygon
aoi = gpd.GeoDataFrame(
    geometry=[box(*bbox)],
    crs="EPSG:28191"
)

# Save AOI
aoi.to_crs("EPSG:4326").to_file(
    data_dir / "Masafer_Yatta_AOI.geojson",
    driver="GeoJSON"
)

print("Bounding box:", bbox)
print("Study area (km²):", aoi.area.iloc[0] / 1e6)

In [ ]:
# Preview tiles form GeoMOLG

from PIL import Image
from io import BytesIO

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (year, url) in zip(axes, services.items()):

    params = {
        "bbox": ",".join(map(str, bbox)),
        "bboxSR": 28191,
        "imageSR": 28191,
        "size": "1500,1500",
        "format": "png24",
        "transparent": "false",
        "f": "image"
    }

    try:
        response = requests.get(
            f"{url}/export",
            params=params,
            timeout=120
        )

        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image = image.convert("RGB")

        image.save(data_dir / f"Preview_{year}.png")

        ax.imshow(image)
        ax.set_title(str(year), fontsize=16)
        ax.axis("off")

        print(f"{year}: Preview successfully downloaded")

    except Exception as e:
        ax.set_title(f"{year}: Download failed")
        ax.axis("off")
        print(f"{year}: {e}")

plt.tight_layout()
plt.show()

In [ ]:
# Download and mosaic orthophotos


from PIL import Image
from io import BytesIO
from rasterio.transform import from_bounds
from rasterio.windows import Window, bounds
from urllib.parse import urljoin

# Download parameters
tile_size = 3334
n_tiles = 2
image_size = tile_size * n_tiles

# Common spatial grid
transform = from_bounds(
    *bbox, image_size, image_size
)

session = requests.Session()

for year, url in services.items():

    output_path = data_dir / f"Orthophoto_{year}.tif"

    print(f"\nDownloading orthophoto {year}...")

    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=image_size,
        width=image_size,
        count=3,
        dtype="uint8",
        crs="EPSG:28191",
        transform=transform,
        compress="lzw",
        photometric="RGB",
        tiled=True
    ) as dst:

        for row in range(n_tiles):
            for col in range(n_tiles):

                # Define tile window
                window = Window(
                    col * tile_size,
                    row * tile_size,
                    tile_size,
                    tile_size
                )

                tile_bbox = bounds(window, transform)

                # GeoMOLG export parameters
                params = {
                    "bbox": ",".join(map(str, tile_bbox)),
                    "bboxSR": 28191,
                    "imageSR": 28191,
                    "size": f"{tile_size},{tile_size}",
                    "format": "png24",
                    "transparent": "false",
                    "f": "json"
                }

                # Request export metadata
                response = session.get(
                    f"{url}/export",
                    params=params,
                    timeout=180
                )
                response.raise_for_status()
                metadata = response.json()

                if "error" in metadata:
                    raise RuntimeError(metadata["error"])

                # Verify returned spatial extent
                extent = metadata["extent"]

                actual_bbox = (
                    extent["xmin"],
                    extent["ymin"],
                    extent["xmax"],
                    extent["ymax"]
                )

                if not np.allclose(
                    actual_bbox, tile_bbox,
                    atol=0.02, rtol=0
                ):
                    raise ValueError("Unexpected tile extent.")

                # Download image
                try:
                    image_response = session.get(
                        urljoin(url + "/", metadata["href"]),
                        timeout=180
                    )
                    image_response.raise_for_status()

                    image = Image.open(
                        BytesIO(image_response.content)
                    ).convert("RGB")

                except Exception:
                    params["f"] = "image"

                    image_response = session.get(
                        f"{url}/export",
                        params=params,
                        timeout=180
                    )
                    image_response.raise_for_status()

                    image = Image.open(
                        BytesIO(image_response.content)
                    ).convert("RGB")

                # Verify image dimensions
                if image.size != (tile_size, tile_size):
                    raise ValueError("Unexpected image dimensions.")

                # Convert RGB image to raster array
                array = np.asarray(image)
                array = np.moveaxis(array, -1, 0)

                # Write tile into output GeoTIFF
                dst.write(array, window=window)

                print(
                    f"  Tile {row * n_tiles + col + 1}/4 completed"
                )

    print(f"Saved: {output_path.name}")

print("\nAll orthophotos downloaded successfully.")


In [ ]:
# Extract test patches
from rasterio.windows import Window
from PIL import Image

patch_size = 1536

patch_dir = data_dir / "test_patches"
patch_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, year in zip(axes, [2021, 2022, 2023]):

    path = data_dir / f"Orthophoto_{year}.tif"

    with rasterio.open(path) as src:

        # Select central study area
        center_x = int(src.width * 0.50)
        center_y = int(src.height * 0.45)

        window = Window(
            center_x - patch_size // 2,
            center_y - patch_size // 2,
            patch_size,
            patch_size
        )

        # Read RGB patch
        rgb = src.read([1, 2, 3], window=window)
        rgb = np.moveaxis(rgb, 0, -1)

    # Save patch for model inference
    output_path = patch_dir / f"Patch_{year}.png"

    Image.fromarray(rgb).save(output_path)

    # Visualization
    ax.imshow(rgb)
    ax.set_title(str(year), fontsize=16)
    ax.axis("off")

    print(f"{year}: Saved {output_path.name}")

plt.tight_layout()
plt.show()

In [ ]:
# Install and load the pre-trained model DINOv3 Buildings Small from HOTOSM

# Install required packages
!pip -q install onnxruntime-gpu huggingface_hub

import numpy as np
import torch
import onnxruntime as ort

from huggingface_hub import hf_hub_download, HfApi

# Model configuration
MODEL_ID = "hotosm/dinov3s-buildings"
MODEL_FILE = "model.onnx"

WINDOW_SIZE = 256
STRIDE = 192
THRESHOLD = 0.4371

# Retrieve model revision for reproducibility
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha

# Download pre-trained model
model_path = hf_hub_download(
    repo_id=MODEL_ID,
    filename=MODEL_FILE,
    revision=MODEL_REVISION
)

# Configure inference device
if torch.cuda.is_available():
    ort.preload_dlls()
    providers = [
        "CUDAExecutionProvider",
        "CPUExecutionProvider"
    ]
else:
    providers = ["CPUExecutionProvider"]

# Initialize ONNX inference session
ort_session = ort.InferenceSession(
    model_path,
    providers=providers
)

# Retrieve model inputs and outputs
model_input = ort_session.get_inputs()[0]
model_output = ort_session.get_outputs()[0]

input_name = model_input.name
output_name = model_output.name

# Display model information
print("Model loaded successfully!")
print("-" * 55)
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Execution providers:", ort_session.get_providers())
print("Input name:", input_name)
print("Input shape:", model_input.shape)
print("Input type:", model_input.type)
print("Output name:", output_name)
print("Output shape:", model_output.shape)
print("Output type:", model_output.type)
print("-" * 55)
print("Window size:", WINDOW_SIZE)
print("Stride:", STRIDE)
print("Probability threshold:", THRESHOLD)

In [ ]:
# Prepare test patches at 15 cm and 30 cm resolution

import cv2
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

years = [2021, 2022, 2023]

# Store test patches at both spatial resolutions
patches = {
    "15cm": {},
    "30cm": {}
}

# Load and resample test patches
for year in years:

    # Load original RGB patch
    rgb_15cm = np.array(
        Image.open(patch_dir / f"Patch_{year}.png").convert("RGB")
    )

    height, width = rgb_15cm.shape[:2]

    # Downsample from 15 cm to 30 cm
    rgb_30cm = cv2.resize(
        rgb_15cm,
        (width // 2, height // 2),
        interpolation=cv2.INTER_AREA
    )

    # Store patches
    patches["15cm"][year] = rgb_15cm
    patches["30cm"][year] = rgb_30cm

    # Save resampled patch
    Image.fromarray(rgb_30cm).save(
        patch_dir / f"Patch_{year}_30cm.png"
    )

    print(f"{year}:")
    print(f"  15 cm: {rgb_15cm.shape}")
    print(f"  30 cm: {rgb_30cm.shape}")

# Visual comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for row, resolution in enumerate(["15cm", "30cm"]):

    for col, year in enumerate(years):

        axes[row, col].imshow(patches[resolution][year])
        axes[row, col].set_title(f"{year} – {resolution}", fontsize=15)
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:

def get_window_positions(length, window_size, stride):
    """
    Calculate sliding-window starting positions.
    Ensures that the final window reaches the image edge.
    """
    if length < window_size:
        raise ValueError("Image dimension is smaller than window size.")

    if stride <= 0:
        raise ValueError("Stride must be positive.")

    positions = list(range(
        0,
        length - window_size + 1,
        stride
    ))

    if positions[-1] != length - window_size:
        positions.append(length - window_size)

    return positions

def preprocess_window(rgb_window):
    """
    Prepare an RGB image window for ONNX inference.
    """
    if rgb_window.shape != (256, 256, 3):
        raise ValueError(
            f"Unexpected window shape: {rgb_window.shape}"
        )

    # Convert RGB values from 0–255 to 0–1
    image = rgb_window.astype(np.float32) / 255.0

    # Convert HWC to CHW and add batch dimension
    image = np.transpose(image, (2, 0, 1))
    image = np.expand_dims(image, axis=0)

    return np.ascontiguousarray(image)

In [ ]:

print("Model input:", model_input.shape)
print("Model type:", model_input.type)

test_window = patches["30cm"][2021][:256, :256]

test_input = preprocess_window(test_window)

print("Prepared shape:", test_input.shape)
print("Prepared dtype:", test_input.dtype)
print("Value range:", test_input.min(), test_input.max())


In [ ]:
# Processing

import numpy as np
import onnxruntime as ort
import time

from scipy.special import expit
from tqdm.auto import tqdm

# Switch to CPU inference
ort_session = ort.InferenceSession(
    model_path,
    providers=["CPUExecutionProvider"]
)

print("Execution providers:", ort_session.get_providers())


# Sliding window inference with numerical validation
def predict_buildings(
    rgb_image,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    threshold=THRESHOLD
):

    height, width = rgb_image.shape[:2]

    probability_sum = np.zeros(
        (height, width), dtype=np.float32
    )

    coverage = np.zeros(
        (height, width), dtype=np.float32
    )

    y_positions = get_window_positions(
        height, window_size, stride
    )

    x_positions = get_window_positions(
        width, window_size, stride
    )

    windows = [
        (y, x)
        for y in y_positions
        for x in x_positions
    ]

    for y, x in tqdm(windows, desc="Inference", leave=False):

        rgb_window = rgb_image[
            y:y + window_size,
            x:x + window_size
        ]

        model_input_array = preprocess_window(rgb_window)

        # ONNX inference
        logits = ort_session.run(
            [output_name],
            {input_name: model_input_array}
        )[0]

        # Check for numerical errors
        if not np.isfinite(logits).all():
            raise ValueError(
                f"Invalid logits at window x={x}, y={y}"
            )

        # Convert building logits to probabilities
        probability = expit(logits[0, 0]).astype(np.float32)

        # Accumulate overlapping predictions
        probability_sum[
            y:y + window_size,
            x:x + window_size
        ] += probability

        coverage[
            y:y + window_size,
            x:x + window_size
        ] += 1

    # Average overlapping predictions
    probability_map = probability_sum / np.maximum(
        coverage, 1
    )

    # Generate binary building mask
    building_mask = (
        probability_map >= threshold
    ).astype(np.uint8)

    return probability_map, building_mask


# Reset previous results
probability_maps = {
    "15cm": {},
    "30cm": {}
}

building_masks = {
    "15cm": {},
    "30cm": {}
}

# Start runtime measurement
start_time = time.time()

# Run inference
for resolution in ["15cm", "30cm"]:

    for year in years:

        print(f"\nProcessing {year} – {resolution}")

        patch_start = time.time()

        probability_map, building_mask = predict_buildings(
            patches[resolution][year]
        )

        probability_maps[resolution][year] = probability_map
        building_masks[resolution][year] = building_mask

        print(f"  Output shape: {building_mask.shape}")
        print(f"  Min probability: {probability_map.min():.4f}")
        print(f"  Max probability: {probability_map.max():.4f}")
        print(f"  NaN values: {np.isnan(probability_map).sum()}")
        print(f"  Building coverage: {building_mask.mean() * 100:.2f}%")
        print(f"  Runtime: {time.time() - patch_start:.2f} s")

print("\nAll test patches successfully processed!")
print(f"Total runtime: {(time.time() - start_time) / 60:.2f} min")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Visualization settings
resolutions = ["15cm", "30cm"]

resolution_labels = {
    "15cm": "15 cm",
    "30cm": "30 cm"
}

building_cmap = ListedColormap(["lime"])

# Check validity of probability maps
for resolution in resolutions:
    for year in years:
        if not np.isfinite(probability_maps[resolution][year]).all():
            raise ValueError(
                f"Invalid probability map: {year} – {resolution}"
            )


# ============================================================
# 1. Original RGB images
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for row, resolution in enumerate(resolutions):
    for col, year in enumerate(years):

        ax = axes[row, col]

        ax.imshow(patches[resolution][year])
        ax.set_title(
            f"{year} – {resolution_labels[resolution]}",
            fontsize=14
        )
        ax.axis("off")

fig.suptitle("Original RGB Orthophotos", fontsize=20)

plt.tight_layout()
plt.show()


# ============================================================
# 2. Building probability maps
# ============================================================

fig, axes = plt.subplots(
    2, 3, figsize=(18, 12),
    layout="constrained"
)

for row, resolution in enumerate(resolutions):
    for col, year in enumerate(years):

        ax = axes[row, col]

        im = ax.imshow(
            probability_maps[resolution][year],
            cmap="viridis",
            vmin=0,
            vmax=1
        )

        ax.set_title(
            f"{year} – {resolution_labels[resolution]}",
            fontsize=14
        )

        ax.axis("off")

cbar = fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    shrink=0.7
)

cbar.set_label("Building probability", fontsize=13)

fig.suptitle("Building Probability Maps", fontsize=20)

plt.show()


# ============================================================
# 3. Building segmentation overlays
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for row, resolution in enumerate(resolutions):
    for col, year in enumerate(years):

        ax = axes[row, col]

        rgb = patches[resolution][year]
        mask = building_masks[resolution][year]

        # Mask background pixels
        overlay = np.ma.masked_where(mask == 0, mask)

        # Plot RGB and building mask
        ax.imshow(rgb)

        ax.imshow(
            overlay,
            cmap=building_cmap,
            alpha=0.45,
            vmin=0,
            vmax=1,
            interpolation="nearest"
        )

        coverage = mask.mean() * 100

        ax.set_title(
            f"{year} – {resolution_labels[resolution]}"
            f"\nBuilding coverage: {coverage:.2f}%",
            fontsize=14
        )

        ax.axis("off")

fig.legend(
    handles=[
        Patch(
            facecolor="lime",
            alpha=0.45,
            label="Detected buildings"
        )
    ],
    loc="lower center",
    fontsize=13
)

fig.suptitle("Building Segmentation Results", fontsize=20)

plt.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap

# Configuration
SELECTED_RESOLUTION = "30cm"

thresholds = [0.20, 0.30, 0.40, 0.4371, 0.50]

building_cmap = ListedColormap(["lime"])

# Store masks and statistics
threshold_masks = {}
threshold_statistics = []

# Create figure
fig, axes = plt.subplots(
    len(thresholds),
    len(years),
    figsize=(18, 27)
)

# Evaluate different thresholds
for row, threshold in enumerate(thresholds):

    threshold_masks[threshold] = {}

    for col, year in enumerate(years):

        # Load original image and probability map
        rgb = patches[SELECTED_RESOLUTION][year]
        probability_map = probability_maps[SELECTED_RESOLUTION][year]

        # Validate probability map
        if not np.isfinite(probability_map).all():
            raise ValueError(f"Invalid probability map: {year}")

        # Apply probability threshold
        mask = (probability_map >= threshold).astype(np.uint8)

        # Store building mask
        threshold_masks[threshold][year] = mask

        # Calculate building statistics
        building_pixels = np.sum(mask)
        building_coverage = mask.mean() * 100
        building_area = building_pixels * 0.30**2

        threshold_statistics.append({
            "Year": year,
            "Threshold": threshold,
            "Building pixels": building_pixels,
            "Building coverage (%)": building_coverage,
            "Building area (m²)": building_area
        })

        # Create transparent building overlay
        overlay = np.ma.masked_where(mask == 0, mask)

        ax = axes[row, col]

        ax.imshow(rgb)

        ax.imshow(
            overlay,
            cmap=building_cmap,
            alpha=0.45,
            vmin=0,
            vmax=1,
            interpolation="nearest"
        )

        ax.set_title(
            f"{year} | Threshold: {threshold:.4f}\n"
            f"Building coverage: {building_coverage:.2f}%",
            fontsize=13
        )

        ax.axis("off")

fig.suptitle(
    "Probability Threshold Sensitivity Analysis (30 cm)",
    fontsize=22
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

# Create statistics dataframe
df_thresholds = pd.DataFrame(threshold_statistics)

# Display building coverage comparison
coverage_table = df_thresholds.pivot(
    index="Threshold",
    columns="Year",
    values="Building coverage (%)"
)

print("\nBuilding coverage (%)")
display(coverage_table.round(3))

# Display building area comparison
area_table = df_thresholds.pivot(
    index="Threshold",
    columns="Year",
    values="Building area (m²)"
)

print("\nEstimated building area (m²)")
display(area_table.round(2))

# Plot threshold sensitivity
fig, ax = plt.subplots(figsize=(10, 5))

for year in years:

    subset = df_thresholds[
        df_thresholds["Year"] == year
    ]

    ax.plot(
        subset["Threshold"],
        subset["Building coverage (%)"],
        marker="o",
        linewidth=2,
        label=str(year)
    )

# Highlight default model threshold
ax.axvline(
    0.4371,
    color="black",
    linestyle="--",
    label="Default threshold"
)

ax.set_xlabel("Probability threshold")
ax.set_ylabel("Building coverage (%)")
ax.set_title("Sensitivity of Building Coverage to Probability Threshold")

ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# Increase of AOI to 1x1 km and resampling


import numpy as np
import rasterio
import matplotlib.pyplot as plt

from rasterio.enums import Resampling
from rasterio.transform import from_bounds

# Configuration
SELECTED_RESOLUTION = "30cm"
SELECTED_THRESHOLD = 0.30
DOWNSCALE_FACTOR = 2

years = [2021, 2022, 2023]

# Create output directory
resampled_dir = data_dir / "orthophotos_30cm"
resampled_dir.mkdir(exist_ok=True)

# Store file paths and RGB arrays
resampled_paths = {}
full_images = {}

# Resample full orthophotos
for year in years:

    input_path = data_dir / f"Orthophoto_{year}.tif"

    output_path = (
        resampled_dir / f"Orthophoto_{year}_30cm.tif"
    )

    with rasterio.open(input_path) as src:

        # Calculate new dimensions
        new_width = src.width // DOWNSCALE_FACTOR
        new_height = src.height // DOWNSCALE_FACTOR

        # Resample RGB bands
        rgb = src.read(
            [1, 2, 3],
            out_shape=(3, new_height, new_width),
            resampling=Resampling.average
        )

        # Preserve original spatial extent
        new_transform = from_bounds(
            *src.bounds,
            new_width,
            new_height
        )

        # Update GeoTIFF metadata
        profile = src.profile.copy()

        profile.update(
            width=new_width,
            height=new_height,
            transform=new_transform,
            count=3,
            compress="lzw"
        )

        # Save resampled GeoTIFF
        with rasterio.open(
            output_path, "w", **profile
        ) as dst:

            dst.write(rgb)

        # Store RGB array in HWC format
        full_images[year] = np.moveaxis(rgb, 0, -1)

        # Store file path
        resampled_paths[year] = output_path

    # Check output metadata
    with rasterio.open(output_path) as dst:

        print(f"\nYear: {year}")
        print(f"  Dimensions: {dst.width} × {dst.height}")
        print(f"  Resolution: {dst.res}")
        print(f"  CRS: {dst.crs}")
        print(f"  Saved: {output_path.name}")


# ============================================================
# Verify spatial consistency
# ============================================================

with rasterio.open(resampled_paths[2021]) as ref:

    reference_crs = ref.crs
    reference_transform = ref.transform
    reference_shape = ref.shape

for year in years:

    with rasterio.open(resampled_paths[year]) as src:

        assert src.crs == reference_crs
        assert src.transform.almost_equals(reference_transform)
        assert src.shape == reference_shape

print("\nAll orthophotos share the same spatial grid.")


# ============================================================
# Visualize resampled orthophotos
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

for ax, year in zip(axes, years):

    ax.imshow(full_images[year])
    ax.set_title(f"{year} – 30 cm", fontsize=15)
    ax.axis("off")

fig.suptitle(
    "Full AOI Orthophotos – 30 cm Resolution",
    fontsize=20
)

plt.tight_layout()
plt.show()

In [ ]:
# Full AOI building segementation
import numpy as np
import rasterio
import pandas as pd
import time

# Configuration
SELECTED_RESOLUTION = "30cm"
SELECTED_THRESHOLD = 0.30

# Create output directory
segmentation_dir = data_dir / "building_segmentation"
segmentation_dir.mkdir(exist_ok=True)

# Store segmentation results
full_probability_maps = {}
full_building_masks = {}

probability_paths = {}
building_mask_paths = {}

segmentation_statistics = []

start_time = time.time()

# Run building segmentation for all years
for year in years:

    print(f"\nProcessing {year} – Full AOI")
    print("-" * 50)

    year_start = time.time()

    # Load RGB orthophoto
    rgb = full_images[year]

    # Sliding window inference
    probability_map, building_mask = predict_buildings(
        rgb,
        window_size=WINDOW_SIZE,
        stride=STRIDE,
        threshold=SELECTED_THRESHOLD
    )

    # Validate probability map
    if not np.isfinite(probability_map).all():
        raise ValueError(f"Invalid probability map: {year}")

    # Retrieve geospatial metadata
    with rasterio.open(resampled_paths[year]) as src:

        profile = src.profile.copy()
        transform = src.transform

    # Calculate actual pixel area
    pixel_area = abs(
        transform.a * transform.e
        - transform.b * transform.d
    )

    # Store arrays
    full_probability_maps[year] = probability_map
    full_building_masks[year] = building_mask

    # Define output paths
    probability_path = (
        segmentation_dir / f"Building_Probability_{year}.tif"
    )

    mask_path = (
        segmentation_dir / f"Building_Mask_{year}.tif"
    )

    # Prepare single-band metadata
    profile.pop("photometric", None)

    probability_profile = profile.copy()

    probability_profile.update(
        count=1,
        dtype="float32",
        nodata=None,
        compress="lzw"
    )

    mask_profile = profile.copy()

    mask_profile.update(
        count=1,
        dtype="uint8",
        nodata=None,
        compress="lzw"
    )

    # Save probability map
    with rasterio.open(
        probability_path, "w", **probability_profile
    ) as dst:

        dst.write(
            probability_map.astype(np.float32), 1
        )

    # Save binary building mask
    with rasterio.open(
        mask_path, "w", **mask_profile
    ) as dst:

        dst.write(
            building_mask.astype(np.uint8), 1
        )

    # Store output paths
    probability_paths[year] = probability_path
    building_mask_paths[year] = mask_path

    # Calculate statistics
    building_pixels = int(building_mask.sum())
    building_coverage = building_mask.mean() * 100
    building_area = building_pixels * pixel_area

    runtime = time.time() - year_start

    segmentation_statistics.append({
        "Year": year,
        "Building pixels": building_pixels,
        "Building coverage (%)": building_coverage,
        "Building area (m²)": building_area,
        "Runtime (min)": runtime / 60
    })

    # Print results
    print(f"Output shape: {building_mask.shape}")
    print(f"Min probability: {probability_map.min():.4f}")
    print(f"Max probability: {probability_map.max():.4f}")
    print(f"Building coverage: {building_coverage:.2f}%")
    print(f"Building area: {building_area:.2f} m²")
    print(f"Runtime: {runtime / 60:.2f} min")
    print("GeoTIFFs successfully saved!")

# Display summary statistics
df_full_segmentation = pd.DataFrame(
    segmentation_statistics
)

print("\nFull AOI segmentation completed!")
print(f"Total runtime: {(time.time() - start_time) / 60:.2f} min")

display(df_full_segmentation.round(3))

In [ ]:
# Visualisation of full AOI Building Segmentation

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Visualization settings
building_cmap = ListedColormap(["lime"])

# Create figure
fig, axes = plt.subplots(
    3, 3,
    figsize=(24, 24),
    layout="constrained"
)

for col, year in enumerate(years):

    rgb = full_images[year]
    probability_map = full_probability_maps[year]
    mask = full_building_masks[year]

    # Calculate building coverage
    coverage = mask.mean() * 100

    # Estimated building area
    area = df_full_segmentation.loc[
        df_full_segmentation["Year"] == year,
        "Building area (m²)"
    ].iloc[0]

    # Row 1: Original orthophoto
    axes[0, col].imshow(rgb)

    axes[0, col].set_title(
        f"{year} – RGB Orthophoto",
        fontsize=16
    )

    # Row 2: Probability map
    im = axes[1, col].imshow(
        probability_map,
        cmap="viridis",
        vmin=0,
        vmax=1
    )

    axes[1, col].set_title(
        f"{year} – Building Probability",
        fontsize=16
    )

    # Row 3: Building segmentation overlay
    overlay = np.ma.masked_where(mask == 0, mask)

    axes[2, col].imshow(rgb)

    axes[2, col].imshow(
        overlay,
        cmap=building_cmap,
        alpha=0.5,
        vmin=0,
        vmax=1,
        interpolation="nearest"
    )

    axes[2, col].set_title(
        f"{year} – Building Segmentation\n"
        f"Area: {area:,.2f} m² | Coverage: {coverage:.2f}%",
        fontsize=16
    )

# Remove axes
for ax in axes.ravel():
    ax.axis("off")

# Probability colorbar
cbar = fig.colorbar(
    im,
    ax=axes[1, :].tolist(),
    shrink=0.7,
    pad=0.01
)

cbar.set_label(
    "Building probability",
    fontsize=14
)

fig.suptitle(
    "Full AOI Building Segmentation (2021–2023)",
    fontsize=24
)

# Save figure
fig.savefig(
    segmentation_dir / "Full_AOI_Building_Segmentation.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# Building area comparison
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

areas = df_full_segmentation["Building area (m²)"]

bars = ax.bar(
    df_full_segmentation["Year"].astype(str),
    areas,
    color=["#5485B4", "#5485B4", "#5485B4"],
    width=0.55
)

ax.bar_label(
    bars,
    labels=[f"{area:,.0f}" for area in areas],
    padding=5,
    fontsize=12
)

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Detected building area (m²)", fontsize=12)

ax.set_title(
    "Detected Building Area (2021–2023)",
    fontsize=15
)

ax.set_ylim(0, areas.max() * 1.15)

ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

fig.savefig(
    segmentation_dir / "Building_Area_Comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Display segmentation statistics
display(df_full_segmentation.round(3))

In [ ]:
# Pixel based building change detection

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Create output directory
change_dir = data_dir / "building_change_detection"
change_dir.mkdir(exist_ok=True)

# Define comparison periods
periods = [
    (2021, 2022),
    (2022, 2023),
    (2021, 2023)
]

# Define change classes
change_classes = {
    0: "Stable background",
    1: "Stable building",
    2: "Building gain",
    3: "Building loss"
}

# Define change colors
change_colors = [
    "#000000",
    "#4886CD",
    "#30B45F",
    "#E04B4B"
]

change_cmap = ListedColormap(change_colors)

# Retrieve geospatial metadata
with rasterio.open(building_mask_paths[2021]) as src:

    profile = src.profile.copy()
    transform = src.transform

# Calculate actual pixel area
pixel_area = abs(
    transform.a * transform.e
    - transform.b * transform.d
)

profile.update(
    count=1,
    dtype="uint8",
    nodata=None,
    compress="lzw"
)

# Store results
change_maps = {}
change_paths = {}
change_statistics = []

# Perform change detection
for year1, year2 in periods:

    period = f"{year1}-{year2}"

    print(f"Processing {period}")

    mask1 = full_building_masks[year1].astype(bool)
    mask2 = full_building_masks[year2].astype(bool)

    assert mask1.shape == mask2.shape

    # Initialize change map
    change_map = np.zeros(
        mask1.shape,
        dtype=np.uint8
    )

    # Stable building
    change_map[mask1 & mask2] = 1

    # Building gain
    change_map[~mask1 & mask2] = 2

    # Building loss
    change_map[mask1 & ~mask2] = 3

    change_maps[period] = change_map

    # Calculate class areas
    stable_area = np.sum(change_map == 1) * pixel_area
    gain_area = np.sum(change_map == 2) * pixel_area
    loss_area = np.sum(change_map == 3) * pixel_area

    net_change = gain_area - loss_area
    gross_change = gain_area + loss_area

    initial_area = mask1.sum() * pixel_area
    final_area = mask2.sum() * pixel_area

    relative_change = (
        net_change / initial_area * 100
        if initial_area > 0 else np.nan
    )

    # Verify area consistency
    assert np.isclose(
        net_change,
        final_area - initial_area
    )

    # Store statistics
    change_statistics.append({
        "Period": period,
        "Stable building (m²)": stable_area,
        "Building gain (m²)": gain_area,
        "Building loss (m²)": loss_area,
        "Gross change (m²)": gross_change,
        "Net change (m²)": net_change,
        "Relative change (%)": relative_change
    })

    # Save georeferenced change map
    output_path = change_dir / f"Building_Change_{period}.tif"

    with rasterio.open(
        output_path, "w", **profile
    ) as dst:

        dst.write(change_map, 1)

        dst.write_colormap(1, {
            0: (0, 0, 0, 255),
            1: (72, 134, 205, 255),
            2: (48, 180, 95, 255),
            3: (224, 75, 75, 255)
        })

    change_paths[period] = output_path

    print(f"  Building gain: {gain_area:.2f} m²")
    print(f"  Building loss: {loss_area:.2f} m²")
    print(f"  Gross change: {gross_change:.2f} m²")
    print(f"  Net change: {net_change:.2f} m²")
    print()


# ============================================================
# Visualize change maps
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(24, 9)
)

for ax, (year1, year2) in zip(axes, periods):

    period = f"{year1}-{year2}"

    change_map = change_maps[period]

    # Display most recent orthophoto
    ax.imshow(full_images[year2])

    # Hide stable background
    overlay = np.ma.masked_where(
        change_map == 0,
        change_map
    )

    ax.imshow(
        overlay,
        cmap=change_cmap,
        vmin=0,
        vmax=3,
        alpha=0.85,
        interpolation="nearest"
    )

    ax.set_title(period, fontsize=17)
    ax.axis("off")

# Create legend
legend_elements = [
    Patch(facecolor=change_colors[1], label="Stable building"),
    Patch(facecolor=change_colors[2], label="Building gain"),
    Patch(facecolor=change_colors[3], label="Building loss")
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=3,
    fontsize=15
)

fig.suptitle(
    "Pixel-Based Building Change Detection",
    fontsize=23
)

plt.tight_layout(rect=[0, 0.06, 1, 0.95])

fig.savefig(
    change_dir / "Building_Change_Comparison.png",
    dpi=250,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# Display and save statistics
# ============================================================

df_change = pd.DataFrame(change_statistics)

display(df_change.round(3))

df_change.to_csv(
    change_dir / "Building_Change_Statistics.csv",
    index=False
)

print("Building change detection completed successfully!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap

# ============================================================
# Simple building change map: 2021 vs 2023
# Background: unchanged / hidden
# Green: building gain
# Red: building loss
# ============================================================

YEAR1 = 2021
YEAR2 = 2023

# Load building masks
mask_2021 = full_building_masks[YEAR1].astype(bool)
mask_2023 = full_building_masks[YEAR2].astype(bool)

# Calculate gain and loss
building_gain = (~mask_2021) & (mask_2023)
building_loss = (mask_2021) & (~mask_2023)
stable_building = mask_2021 & mask_2023

# Create simple change map
# 0 = background / unchanged
# 1 = stable building
# 2 = building gain
# 3 = building loss
simple_change_map = np.zeros(mask_2021.shape, dtype=np.uint8)
simple_change_map[stable_building] = 1
simple_change_map[building_gain] = 2
simple_change_map[building_loss] = 3

# Calculate area statistics
gain_pixels = np.sum(building_gain)
loss_pixels = np.sum(building_loss)
stable_pixels = np.sum(stable_building)

gain_area = gain_pixels * pixel_area
loss_area = loss_pixels * pixel_area
stable_area = stable_pixels * pixel_area
net_change = gain_area - loss_area

# Print summary
summary_df = pd.DataFrame([{
    "Comparison": "2021 vs 2023",
    "Stable building (m²)": stable_area,
    "Building gain (m²)": gain_area,
    "Building loss (m²)": loss_area,
    "Net change (m²)": net_change
}])

display(summary_df.round(2))

# ============================================================
# Visualization
# ============================================================

fig, ax = plt.subplots(figsize=(12, 12))

# Show 2023 orthophoto as background
ax.imshow(full_images[YEAR2])

# Show only gain and loss as overlay
overlay = np.zeros((*simple_change_map.shape, 4), dtype=np.float32)

# Stable buildings in blue (optional, lighter)
overlay[simple_change_map == 1] = [0.2, 0.5, 0.9, 0.35]

# Gain in green
overlay[simple_change_map == 2] = [0.1, 0.9, 0.2, 0.8]

# Loss in red
overlay[simple_change_map == 3] = [0.9, 0.1, 0.1, 0.8]

ax.imshow(overlay)

ax.set_title(
    "2023 Orthophoto with Building Gain and Loss Compared to 2021\n"
    f"Gain: {gain_area:.1f} m² | Loss: {loss_area:.1f} m² | Net change: {net_change:.1f} m²",
    fontsize=14
)

ax.axis("off")

legend_elements = [
    Patch(facecolor=(0.2, 0.5, 0.9, 0.7), label="Stable building"),
    Patch(facecolor=(0.1, 0.9, 0.2, 0.8), label="Building gain"),
    Patch(facecolor=(0.9, 0.1, 0.1, 0.8), label="Building loss")
]

ax.legend(
    handles=legend_elements,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=3,
    frameon=True
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import Patch

# Configuration
CENTER_X = 1667
CENTER_Y = 1667

ZOOM_SIZE = 768

# Calculate image boundaries
height, width = simple_change_map.shape

x_min = max(0, CENTER_X - ZOOM_SIZE // 2)
x_max = min(width, CENTER_X + ZOOM_SIZE // 2)

y_min = max(0, CENTER_Y - ZOOM_SIZE // 2)
y_max = min(height, CENTER_Y + ZOOM_SIZE // 2)

# Extract orthophoto subsets
rgb_2021 = full_images[2021][y_min:y_max, x_min:x_max]
rgb_2023 = full_images[2023][y_min:y_max, x_min:x_max]

# Extract change map subset
change_subset = simple_change_map[
    y_min:y_max, x_min:x_max
]

# Create transparent overlay
overlay = np.zeros(
    (*change_subset.shape, 4),
    dtype=np.float32
)

# Stable buildings
overlay[change_subset == 1] = [0.2, 0.5, 0.9, 0.35]

# Building gain
overlay[change_subset == 2] = [0.1, 0.9, 0.2, 0.8]

# Building loss
overlay[change_subset == 3] = [0.9, 0.1, 0.1, 0.8]

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(21, 8))

axes[0].imshow(rgb_2021)
axes[0].set_title("2021 – Orthophoto", fontsize=15)

axes[1].imshow(rgb_2023)
axes[1].set_title("2023 – Orthophoto", fontsize=15)

axes[2].imshow(rgb_2023)
axes[2].imshow(overlay)
axes[2].set_title("2021–2023 – Building Change", fontsize=15)

for ax in axes:
    ax.axis("off")

# Legend
legend_elements = [
    Patch(facecolor="#4886CD", label="Stable building"),
    Patch(facecolor="#30B45F", label="Building gain"),
    Patch(facecolor="#E04B4B", label="Building loss")
]

fig.legend(
    handles=legend_elements,
    loc="lower center",
    ncol=3,
    fontsize=13
)

fig.suptitle(
    "Detailed Building Change Assessment – Masafer Yatta",
    fontsize=20
)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])

fig.savefig(
    change_dir / "Detailed_Building_Change_2021-2023.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()